# Bao phase-transition data audit

Upload or mount a JSONL artifact generated by the Node.js research runner. This notebook performs structural checks and basic descriptive summaries; it does not implement Bao rules.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path('/content/phase-transition-observations.jsonl')
assert DATA_PATH.exists(), f'Upload the JSONL file to {DATA_PATH}'

In [ ]:
records = []
with DATA_PATH.open(encoding='utf-8') as source:
    for line_number, line in enumerate(source, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f'Invalid JSON at line {line_number}: {exc}') from exc

df = pd.json_normalize(records)
print(f'rows={len(df):,}, columns={len(df.columns)}')
df.head()

In [ ]:
required = {
    'schemaVersion', 'ply', 'player', 'phase', 'stateHash',
    'legalMoveCount', 'captureMoveCount', 'forcedCapture'
}
missing_columns = sorted(required - set(df.columns))
assert not missing_columns, f'Missing required columns: {missing_columns}'
assert df['stateHash'].str.fullmatch(r'[a-f0-9]{64}').all(), 'Invalid state hash found'
assert not df.duplicated(subset=[c for c in ['gameId', 'conditionId', 'ply'] if c in df]).any(), 'Duplicate observations found'
print('basic audit passed')

In [ ]:
summary_columns = [c for c in [
    'phase', 'legalMoveCount', 'captureMoveCount', 'nonCaptureMoveCount',
    'boardSeedCount', 'nonEmptyPitCount', 'frontRow.occupancyRate.0',
    'frontRow.occupancyRate.1'
] if c in df]
df[summary_columns].describe(include='all')

In [ ]:
if 'gameId' in df and 'ply' in df:
    display(df.groupby('gameId')['ply'].agg(['min', 'max', 'count']).head(20))
display(df.groupby('phase').size().rename('observations'))